# T39 - 成交数据
## 第二章：数据采集

本章介绍成交数据的采集流程，包括：
1. 评级狗API认证
2. 成交数据查询
3. 分页数据获取
4. 数据存储到数据库

## 1. 导入必要的库和配置

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from time import sleep
import logging
from typing import Optional, Tuple, List

# 数据库连接
import sqlalchemy
from sqlalchemy.sql import text

# HTTP请求
import requests

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('数据采集')

print('导入成功！')

## 2. API认证模块

In [ ]:
class RatingDogAuthenticator:
    """评级狗API认证器"""
    
    def __init__(self):
        self.access_token = None
        self.token_expires = None
    
    def login(self) -> Tuple[bool, str]:
        """
        登录评级狗API获取访问令牌
        
        Returns:
            Tuple[bool, str]: (是否成功, 消息)
        """
        try:
            response = requests.post(
                config.ratingdog_api.login_url,
                headers=config.ratingdog_api.get_login_headers(),
                json=config.ratingdog_api.get_login_data(),
                timeout=config.ratingdog_api.timeout
            )
            
            result = response.json()
            
            if result.get('success'):
                self.access_token = result['result']['accessToken']
                # 令牌有效期通常为2小时
                self.token_expires = datetime.now() + timedelta(hours=2)
                logger.info('评级狗API登录成功')
                return True, '登录成功'
            else:
                error_msg = result.get('error', {}).get('message', '未知错误')
                logger.error(f'登录失败: {error_msg}')
                return False, error_msg
                
        except requests.exceptions.Timeout:
            return False, '请求超时'
        except requests.exceptions.ConnectionError:
            return False, '连接错误'
        except Exception as e:
            logger.exception('登录异常')
            return False, str(e)
    
    def get_valid_token(self) -> Optional[str]:
        """
        获取有效的访问令牌，如果过期则重新登录
        
        Returns:
            Optional[str]: 访问令牌或None
        """
        if self.access_token and self.token_expires and datetime.now() < self.token_expires:
            return self.access_token
        
        success, _ = self.login()
        return self.access_token if success else None

# 创建认证器实例
authenticator = RatingDogAuthenticator()

# 测试登录
success, msg = authenticator.login()
print(f'登录结果: {msg}')

## 3. 数据采集核心模块

In [ ]:
class TradeDataCollector:
    """成交数据采集器"""
    
    def __init__(self, authenticator: RatingDogAuthenticator):
        self.authenticator = authenticator
        self.db_engine = None
    
    def _get_db_engine(self):
        """获取数据库引擎"""
        if self.db_engine is None:
            self.db_engine = sqlalchemy.create_engine(
                config.database.get_mysql_yq_connection_string(),
                poolclass=sqlalchemy.pool.NullPool
            )
        return self.db_engine
    
    def _make_request_with_retry(self, post_dict: dict) -> Optional[dict]:
        """
        带重试机制的请求
        
        Args:
            post_dict: 请求参数
            
        Returns:
            Optional[dict]: 响应数据或None
        """
        access_token = self.authenticator.get_valid_token()
        if not access_token:
            logger.error('无法获取有效的访问令牌')
            return None
        
        headers = config.ratingdog_api.get_api_headers(access_token)
        
        for attempt in range(config.ratingdog_api.max_retries):
            try:
                response = requests.post(
                    config.ratingdog_api.trade_data_url,
                    headers=headers,
                    json=post_dict,
                    timeout=config.ratingdog_api.timeout
                )
                
                result = response.json()
                
                if result.get('success'):
                    return result
                else:
                    logger.warning(f'请求失败(尝试 {attempt+1}/{config.ratingdog_api.max_retries}): {result.get("error", {}).get("message", "未知错误")}')
                    
            except requests.exceptions.Timeout:
                logger.warning(f'请求超时(尝试 {attempt+1}/{config.ratingdog_api.max_retries})')
            except requests.exceptions.ConnectionError:
                logger.warning(f'连接错误(尝试 {attempt+1}/{config.ratingdog_api.max_retries})')
            except Exception as e:
                logger.warning(f'请求异常(尝试 {attempt+1}/{config.ratingdog_api.max_retries}): {e}')
            
            # 重试前等待
            if attempt < config.ratingdog_api.max_retries - 1:
                sleep(config.ratingdog_api.retry_interval * (attempt + 1))  # 指数退避
        
        logger.error('达到最大重试次数，请求失败')
        return None
    
    def fetch_trade_data(self, start_date: str, end_date: str) -> pd.DataFrame:
        """
        获取指定日期范围的成交数据
        
        Args:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
            
        Returns:
            pd.DataFrame: 成交数据
        """
        all_data = []
        skip_count = 0
        total_count = None
        
        while True:
            # 构建请求参数
            post_dict = {
                "BondTypes": [],
                "Citys": [],
                "EndTradedDate": end_date,
                "IssueMethods": [],
                "IssuerTypes": [],
                "MaxResultCount": config.ratingdog_api.max_result_count,
                "Natures": [],
                "SkipCount": skip_count,
                "SourceTypes": [],
                "StartTradedDate": start_date,
                "TransactionMarkets": [],
                "YYIndustrys": [],
                "YYRatings": [],
            }
            
            # 发送请求
            result = self._make_request_with_retry(post_dict)
            
            if result is None:
                break
            
            # 获取总数
            if total_count is None:
                total_count = result['result'].get('totalCount', 0)
                logger.info(f'总记录数: {total_count}')
            
            # 提取数据
            items = result['result'].get('items', [])
            if not items:
                break
            
            all_data.extend(items)
            
            # 计算进度
            page_num = skip_count // config.ratingdog_api.max_result_count
            progress = min(100, (len(all_data) / total_count * 100) if total_count > 0 else 0)
            logger.info(f'第{page_num}页完成，进度: {progress:.1f}% ({len(all_data)}/{total_count})')
            
            # 检查是否还有更多数据
            if len(items) < config.ratingdog_api.max_result_count:
                break
            
            skip_count += config.ratingdog_api.max_result_count
            
            # 添加延迟，避免请求过快
            sleep(0.5)
        
        # 转换为DataFrame
        df = pd.DataFrame(all_data)
        logger.info(f'共获取 {len(df)} 条记录')
        
        return df
    
    def save_to_database(self, df: pd.DataFrame) -> Tuple[bool, str]:
        """
        将数据保存到数据库
        
        Args:
            df: 要保存的数据
            
        Returns:
            Tuple[bool, str]: (是否成功, 消息)
        """
        if df.empty:
            return True, '没有数据需要保存'
        
        try:
            engine = self._get_db_engine()
            df.to_sql(
                config.process.output_table,
                engine,
                if_exists='append',
                index=False
            )
            logger.info(f'成功保存 {len(df)} 条记录到数据库')
            return True, f'保存成功，共{len(df)}条'
        except Exception as e:
            logger.exception('保存数据失败')
            return False, str(e)

# 创建采集器实例
collector = TradeDataCollector(authenticator)
print('采集器初始化完成')

## 4. 获取待采集日期列表

In [ ]:
def get_pending_dates() -> List[str]:
    """
    获取待采集的日期列表
    
    从bond.marketinfo_abs表中获取所有日期，排除yq.cjqb中已存在的日期
    
    Returns:
        List[str]: 待采集日期列表
    """
    try:
        # 连接bond数据库查询日期
        bond_engine = sqlalchemy.create_engine(
            config.database.get_mysql_bond_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        
        # 连接yq数据库检查已存在的日期
        yq_engine = sqlalchemy.create_engine(
            config.database.get_mysql_yq_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        
        # 查询待采集日期
        query = text('''
            SELECT DISTINCT dt 
            FROM bond.marketinfo_abs 
            WHERE dt NOT IN (SELECT DISTINCT dt FROM yq.cjqb)
            ORDER BY dt
        ''')
        
        # 使用bond连接执行跨库查询
        with bond_engine.connect() as conn:
            result = conn.execute(query)
            dates = [row[0] for row in result.fetchall()]
        
        logger.info(f'待采集日期数: {len(dates)}')
        if dates:
            logger.info(f'日期范围: {dates[0]} ~ {dates[-1]}')
        
        return dates
        
    except Exception as e:
        logger.exception('获取待采集日期失败')
        return []

# 获取待采集日期
pending_dates = get_pending_dates()
print(f'\n待采集日期数: {len(pending_dates)}')
if pending_dates:
    print(f'前5个日期: {pending_dates[:5]}')
    print(f'后5个日期: {pending_dates[-5:]}')

## 5. 执行数据采集

In [ ]:
def collect_data_for_dates(dates: List[str], max_dates: int = None):
    """
    批量采集成交数据
    
    Args:
        dates: 日期列表
        max_dates: 最大采集日期数（用于测试，None表示全部）
    """
    if not dates:
        print('没有待采集的日期')
        return
    
    # 限制采集数量（用于测试）
    if max_dates:
        dates = dates[:max_dates]
        print(f'测试模式：只采集前 {max_dates} 个日期')
    
    total = len(dates)
    success_count = 0
    fail_count = 0
    
    print(f'\n开始采集，共 {total} 个日期')
    print('='*60)
    
    for i, date in enumerate(dates, 1):
        print(f'\n[{i}/{total}] 正在采集: {date}')
        
        try:
            # 获取当天数据
            df = collector.fetch_trade_data(date, date)
            
            if df.empty:
                print(f'  日期 {date} 无数据')
                success_count += 1
                continue
            
            # 保存到数据库
            success, msg = collector.save_to_database(df)
            
            if success:
                print(f'  成功: {msg}')
                success_count += 1
            else:
                print(f'  失败: {msg}')
                fail_count += 1
                
        except Exception as e:
            print(f'  异常: {e}')
            fail_count += 1
    
    print('\n' + '='*60)
    print('采集完成！')
    print(f'成功: {success_count}, 失败: {fail_count}')

# 执行采集（测试模式：只采集前3个日期）
# 如需完整采集，将max_dates设为None
collect_data_for_dates(pending_dates, max_dates=3)

## 6. 数据采集统计

In [ ]:
def show_collection_stats():
    """显示数据采集统计"""
    try:
        engine = sqlalchemy.create_engine(
            config.database.get_mysql_yq_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        
        with engine.connect() as conn:
            # 总记录数
            result = conn.execute(text('SELECT COUNT(*) FROM yq.cjqb'))
            total_records = result.fetchone()[0]
            
            # 日期数
            result = conn.execute(text('SELECT COUNT(DISTINCT dt) FROM yq.cjqb'))
            date_count = result.fetchone()[0]
            
            # 日期范围
            result = conn.execute(text('SELECT MIN(dt), MAX(dt) FROM yq.cjqb'))
            row = result.fetchone()
            min_date, max_date = row[0], row[1]
            
            # 债券数
            result = conn.execute(text('SELECT COUNT(DISTINCT bondCode) FROM yq.cjqb'))
            bond_count = result.fetchone()[0]
        
        print('\n=== 成交数据采集统计 ===')
        print(f'总记录数: {total_records:,}')
        print(f'日期数: {date_count}')
        print(f'日期范围: {min_date} ~ {max_date}')
        print(f'债券数: {bond_count:,}')
        
    except Exception as e:
        print(f'获取统计失败: {e}')

show_collection_stats()

## 7. 数据预览

In [ ]:
def preview_data(limit=10):
    """预览成交数据"""
    try:
        engine = sqlalchemy.create_engine(
            config.database.get_mysql_yq_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        
        df = pd.read_sql(f'SELECT * FROM yq.cjqb ORDER BY tradedDate DESC LIMIT {limit}', engine)
        
        print(f'\n=== 最近{limit}条成交数据 ===')
        print(df.to_string())
        
    except Exception as e:
        print(f'预览数据失败: {e}')

preview_data(5)